In [1]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA, AirQuality
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.nn import models
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from torch.optim.lr_scheduler import MultiStepLR
from pytorch_lightning.loggers import TensorBoardLogger
from utils import MaskedRMSE
from dataset_utils import SDWPE
import numpy as np
import torch
import torch.nn as nn
import torch.sparse
from einops import rearrange
from torch.nn import functional as F
from tsl.nn.utils import get_functional_activation
from tqdm import tqdm
import torch
from tsl.nn.layers.graph_convs import DiffConv

def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    # if torch.cuda.is_available():
    #     torch.cuda.manual_seed(seed)
    #     torch.cuda.manual_seed_all(seed)
    #     torch.backends.cudnn.deterministic = True
    #     torch.backends.cudnn.benchmark = True
    #     torch.set_float32_matmul_precision('medium')  # 'medium' favors performance over precision

    #     # Enable TF32 format which is optimized for Tensor Cores on Ampere+ GPUs
    #     torch.backends.cuda.matmul.allow_tf32 = True
    #     torch.backends.cudnn.allow_tf32 = True
        
    # os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(42)

42

In [2]:
dataset = MetrLA(root='./data/metrla')

connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        # normalize_axis=1,
                                        force_symmetric=False,
                                        layout="edge_index")

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                      mask=dataset.mask,
                                      covariates=covariates,
                                      horizon=12,
                                      window=1,
                                      stride=1)
print(torch_dataset)

SpatioTemporalDataset(n_samples=34260, n_nodes=207, n_channels=1)


In [3]:
# dataset = AirQuality(root='./data/aq', impute_nans=True, small=False)

# splitting = {"val_len": 0.1,
#             "test_len": 0.2}


# connectivity_sparse= {"method": "distance",
#                     "threshold": 0.1,
#                     "include_self": False,
#                     "layout": "edge_index"}

# adj = dataset.get_connectivity(**connectivity_sparse)

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
#                                       connectivity=adj,
#                                       mask=dataset.mask,
#                                       covariates=covariates,
#                                       horizon=12,
#                                       window=12,
#                                       stride=1)

# torch_dataset

In [4]:
# dataset = SDWPE()

# splitting = {"val_len": 0.1,
#             "test_len": 0.2}


# connectivity_sparse= {"method": "distance",
#                     "threshold": 0.1,
#                     "include_self": False,
#                     "layout": "edge_index"}

# adj = dataset.get_connectivity(**connectivity_sparse)

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
#                                       connectivity=adj,
#                                       mask=dataset.mask,
#                                       covariates=covariates,
#                                        window=1,
#                                       horizon=12,
#                                       stride=1)

# torch_dataset

In [5]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=16,
    # workers = 4
)

dm.setup()
print(dm)

{Train dataloader: size=24667}
{Validation dataloader: size=2739}
{Test dataloader: size=6852}
{Predict dataloader: None}


## Utils

In [6]:
from typing import (Any, Callable, List, Mapping, Optional, Sequence, Set,
                    Type, Union)


def ensure_list(value: Any) -> List:
    # if isinstance(value, Sequence) and not isinstance(value, str):
    if hasattr(value, '__iter__') and not isinstance(value, str):
        return list(value)
    else:
        return [value]



def encode_dataset(
        dataset,
        encoder_class,
        encode_exogenous=True,
        keep_raw=False,
        save_path=None
):
    if encode_exogenous:
        x, _ = dataset.collate_keys(['target', 'u'], preprocess=True, cat_dim=-1)
    else:
        x, _ = dataset.collate_keys(['target'], preprocess=True, cat_dim=-1)

    encoder = encoder_class.to('cuda')

    encoded_x = encoder(x.to('cuda'), edge_index=dataset.edge_index.to('cuda'), edge_weight=dataset.edge_weight.to('cuda'))
    
    print(encoded_x.shape)

    if save_path is not None:
        torch.save(encoded_x, save_path)

    dataset.add_exogenous('encoded_x', encoded_x, add_to_input_map=False)

    input_map = {'x': ['encoded_x']}
    u = ([] if encode_exogenous else ['u']) + (['data'] if keep_raw else [])
    if len(u):
        input_map['u'] = u
    dataset.set_input_map(input_map)

    dataset = dataset
    torch.cuda.empty_cache()
    gc.collect()

    
    return dataset


## SGP

In [7]:
class ReservoirLayer(nn.Module):
    def __init__(self,
                 input_size,
                 hidden_size,
                 spectral_radius,
                 leaking_rate,
                 bias=True,
                 density=1.,
                 in_scaling=1.,
                 bias_scale=1.,
                 activation='tanh'):
        super(ReservoirLayer, self).__init__()
        self.w_ih_scale = in_scaling
        self.b_scale = bias_scale
        self.density = density
        self.hidden_size = hidden_size
        self.alpha = leaking_rate
        self.spectral_radius = spectral_radius

        assert activation in ['tanh', 'relu']
        self.activation = get_functional_activation(activation)

        self.w_ih = nn.Parameter(torch.Tensor(hidden_size, input_size),
                                 requires_grad=False)
        self.w_hh = nn.Parameter(torch.Tensor(hidden_size, hidden_size),
                                 requires_grad=False)
        if bias is not None:
            self.b_ih = nn.Parameter(torch.Tensor(hidden_size),
                                     requires_grad=False)
        else:
            self.register_parameter('b_ih', None)
        self.reset_parameters()

    def reset_parameters(self):
        self.w_ih.data.uniform_(-1, 1)
        self.w_ih.data.mul_(self.w_ih_scale)

        if self.b_ih is not None:
            self.b_ih.data.uniform_(-1, 1)
            self.b_ih.data.mul_(self.b_scale)

        # init recurrent weights
        self.w_hh.data.uniform_(-1, 1)

        if self.density < 1:
            n_units = self.hidden_size * self.hidden_size
            mask = self.w_hh.data.new_ones(n_units)
            masked_weights = torch.randperm(n_units)[
                             :int(n_units * (1 - self.density))]
            mask[masked_weights] = 0.
            self.w_hh.data.mul_(mask.view(self.hidden_size, self.hidden_size))

        # adjust spectral radius
        abs_eigs = torch.linalg.eigvals(self.w_hh.data).abs()
        self.w_hh.data.mul_(self.spectral_radius / torch.max(abs_eigs))

    def forward(self, x, h):
        h_new = self.activation(
            F.linear(x, self.w_ih, self.b_ih) + F.linear(h, self.w_hh))
        h_new = (1 - self.alpha) * h + self.alpha * h_new
        return h_new


class Reservoir(nn.Module):
    def __init__(self,
                 input_size,
                 hidden_size,
                 input_scaling=1.,
                 num_layers=1,
                 leaking_rate=0.9,
                 spectral_radius=0.9,
                 density=0.9,
                 activation='tanh',
                 bias=True,
                 alpha_decay=False):
        super(Reservoir, self).__init__()
        self.mode = activation
        self.input_size = input_size
        self.input_scaling = input_scaling
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.leaking_rate = leaking_rate
        self.spectral_radius = spectral_radius
        self.density = density
        self.bias = bias
        self.alpha_decay = alpha_decay

        layers = []
        alpha = leaking_rate
        for i in range(num_layers):
            layers.append(
                ReservoirLayer(
                    input_size=input_size if i == 0 else hidden_size,
                    hidden_size=hidden_size,
                    in_scaling=input_scaling,
                    density=density,
                    activation=activation,
                    spectral_radius=spectral_radius,
                    leaking_rate=alpha
                )
            )
            if self.alpha_decay:
                alpha = np.clip(alpha - 0.1, 0.1, 1.)

        self.reservoir_layers = nn.ModuleList(layers)

    def reset_parameters(self):
        for layer in self.reservoir_layers:
            layer.reset_parameters()

            
    def forward(self, x, h0=None, return_last_state=False):
        # x : b s n f
        batch_size, steps, nodes, _ = x.size()

        if h0 is None:
            h0 = x.new_zeros(len(self.reservoir_layers), batch_size * nodes,
                             self.hidden_size, requires_grad=False)

        x = rearrange(x, 'b s n f -> s (b n) f')
        out = []
        h = h0
        # for each step, update the reservoir states for all layers
        for s in tqdm(range(steps)):
            h_s = []
            # for all layers, observe input and compute updated states
            x_s = x[s]
            for i, layer in enumerate(self.reservoir_layers):
                x_s = layer(x_s, h[i])
                h_s.append(x_s)
            # update all states
            h = torch.stack(h_s)
            # collect states
            out.append(h.cpu())
        out = torch.stack(out)  # [s, l, b, (n), f]
        out = rearrange(out, 's l (b n) f -> b s n (l f)', b=batch_size,
                        n=nodes)
        print('finished temporal encoding')
        if return_last_state:
            return out[:, -1]
        return out


In [8]:
import torch
from tsl.nn.layers.graph_convs import DiffConv


class IdentityDiffConv(DiffConv):
    """DiffConv with identity filters (no learnable parameters)."""
    
    def __init__(self, in_channels, out_channels=1, k=3, **kwargs):
        super().__init__(in_channels, out_channels, k, bias=False, **kwargs)
        del self.filters  # Remove learnable layer
    
    def forward(self, x, edge_index, edge_weight=None, cache_support=False):
        # Reuse parent's diffusion logic
        n = x.size(-2)
        if self._support is None:
            support = self.compute_support_index(
                edge_index, edge_weight, 
                add_backward=self.add_backward, num_nodes=n)
            if cache_support:
                self._support = support
        else:
            support = self._support

        out = [x] if self.root_weight else []
        
        for sup_index, sup_weights in support:
            x_sup = x
            for _ in range(self.k):
                x_sup = self.propagate(sup_index, x=x_sup, weight=sup_weights)
                out.append(x_sup)

        out = torch.cat(out, -1)
        return out

In [9]:
class SGPEncoder(nn.Module):
    def __init__(self,
                 input_size = 3,
                 reservoir_size = 64,
                 reservoir_layers = 4,
                 leaking_rate = 0.9,
                 spectral_radius = 0.9,
                 density = 0.7,
                 input_scaling = 1.,
                 alpha_decay = False,
                 reservoir_activation='tanh'
                 ):

        super(SGPEncoder, self).__init__()
        self.reservoir = Reservoir(input_size=input_size,
                                   hidden_size=reservoir_size,
                                   input_scaling=input_scaling,
                                   num_layers=reservoir_layers,
                                   leaking_rate=leaking_rate,
                                   spectral_radius=spectral_radius,
                                   density=density,
                                   activation=reservoir_activation,
                                   alpha_decay=alpha_decay)

        self.sgp_encoder = IdentityDiffConv(in_channels=reservoir_size*reservoir_layers)

    def forward(self, x, edge_index, edge_weight):
        # x : [t n f]
        x = rearrange(x, 't n f -> 1 t n f')
        x = self.reservoir(x)
        x = x[0]
        x = self.sgp_encoder(x, edge_index, edge_weight)
        return x


In [10]:
encoder_cls = SGPEncoder()

encode_dataset(torch_dataset, encoder_cls)

100%|████████████████████████████████████████████████████████████████████████████████| 34272/34272 [01:27<00:00, 393.91it/s]


finished temporal encoding


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [6]:
from tsl.nn.blocks.encoders import DCRNN, ConditionalBlock
from tsl.nn.blocks.encoders import DCRNN

# Inherit from your original DCRNNModel
class CustomDCRNNModel(models.DCRNNModel):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        # Replace only the dcrnn layer
        self.dcrnn = DCRNN(input_size=self.dcrnn.input_size,
                           hidden_size=self.dcrnn.hidden_size,
                           n_layers=len(self.dcrnn.cells),
                           k=self.dcrnn.k,
                           return_only_last_state=True,
                           root_weight=False,
                           add_backward=True)

In [7]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
        'mae': torch_metrics.MaskedMAE(),
        'rmse': MaskedRMSE(),
        # 'mae_step_2': torch_metrics.MaskedMAE(at=2),
        # 'mae_step_3': torch_metrics.MaskedMAE(at=5),
        # 'mae_step_4': torch_metrics.MaskedMAE(at=11),
        # 'mse_step_2': torch_metrics.MaskedMSE(at=2),
        # 'mse_step_3': torch_metrics.MaskedMSE(at=5),
        # 'mse_step_4': torch_metrics.MaskedMSE(at=11)
    }

model = CustomDCRNNModel(input_size=1,exog_size=2, hidden_size = 64, output_size=1,
                          horizon=12, ff_size = 128, dropout = 0.1,kernel_size=3,
                          cache_support=True, n_layers = 2)

def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)

In [8]:
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-4
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)
# 'momentum':0.9,
#                  'nesterov':True

In [9]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f'model_checkpoint/{dataset.name}/{model.__class__.__name__}',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
    verbose=True,
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=5,
        mode='min',
    min_delta = 0.001
    )

trainer = Trainer(
        max_epochs=200,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[1],
        gradient_clip_val=5,
       callbacks=[early_stop_callback],
      # default_root_dir="logs",
        # profiler=profiler,
        # precision = '32',
        check_val_every_n_epoch = 3,
        logger=False

    
)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [10]:
trainer.fit(predictor, datamodule=dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | CustomDCRNNModel | 313 K  | train
-----------------------------------------------------------
313 K     Trainable params
0         Non-trainable params
313 K     Total params
1.255     Total estimated model params size (MB)
49        Modules in train mode
0         Modules in eval mode


Training: |                                                                                           | 0/? [0…

Only args ['u', 'edge_weight', 'x', 'edge_index'] are forwarded to the model (CustomDCRNNModel).


Validation: |                                                                                         | 0/? [0…

Validation: |                                                                                         | 0/? [0…

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

Validation: |                                                                 | 0/? [00:00<?, ?it/s]

In [11]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints/epoch=20-step=3150-v1.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints/epoch=20-step=3150-v1.ckpt


Testing: |                                                                    | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     88.9886474609375      │
│         test_mae          │     91.8371353149414      │
│         test_rmse         │     165.9429473876953     │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 91.8371353149414,
  'test_rmse': 165.9429473876953,
  'test_loss': 88.9886474609375}]

<table>
  <tr>
    <th>Model</th>
    <th colspan="2" align="center">Metr LA</th>
    <th colspan="2" align="center">AirQuality 36</th>
      <th colspan="2" align="center">AirQuality Full</th>
  </tr>
  <tr>
    <th></th>
    <th>MAE</th>
    <th>MSE</th>
    <th>MAE</th>
    <th>MSE</th>
    <th>MAE</th>
    <th>MSE</th>
  </tr>
  <tr>
    <td>DCRNN Directed</td>
    <td>3.18</td>
    <td>39.67</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
  </tr>
    <tr>
    <td>DCRNN Undirected</td>
    <td>3.27</td>
    <td>42.04</td>
    <td>31.96</td>
    <td>2593.73</td>
    <td>21.21</td>
    <td>1414.68</td>
  </tr>
  <tr>
    <td>Graph Wavenet Directed</td>
    <td>3.16</td>
    <td>38.88</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
    <td>-</td>
  </tr>
    <tr>
    <td>Graph Wavenet undirected</td>
    <td>3.24</td>
    <td>41.09</td>
    <td>30.63</td>
    <td>2344.78</td>
    <td>21.07</td>
    <td>1380.89</td>
  </tr>
  <tr>
    <td>Ours</td>
    <td>3.83</td>
    <td>59.78</td>
    <td>33.12</td>
    <td>2699.32</td>
    <td>23.12</td>
    <td>1558.18</td>
  </tr>
</table>